In [1]:
import scanpy as sc
import pandas as pd

In [5]:
adata = sc.read("../data/h5ad_files/FINAL_int_retac_adata.h5ad")

In [6]:
adata.obs["cell_type"] = adata.obs[["AQ_int_cell_type", "adult_rlt_cell_type", "brown_cell_type", "son_cell_type", "JW_cell_type", "littman_cell_type"]].bfill(axis=1).iloc[:, 0]

In [7]:
adata.obs["cell_type"].value_counts()

cell_type
R-eTAC1            2635
JC1                1861
R-cDC2             1688
R-eTAC2            1193
R-mDC              1152
R-eTAC3             995
eTACs I             821
JC2                 706
R-cDC1              597
TC IV               539
LTi-like R-eTAC     428
TC I                381
TC II               345
TC III              251
Pro. R-eTAC         139
Drop                118
eTAC IIs             78
Name: count, dtype: int64

In [8]:
adata.obs["dataset"] = adata.obs.final_sample

In [9]:
sc.pp.normalize_total(adata, target_sum=10000)
sc.pp.log1p(adata)

In [10]:
umap = pd.DataFrame(adata.obsm["X_umap"], index=adata.obs_names, columns=["x", "y"])

In [11]:
log1p = pd.DataFrame(adata.X.todense(), index=adata.obs_names, columns=adata.var_names)
log1p = pd.concat([log1p, adata.obs[["dataset", "cell_type"]], umap], axis=1)

In [12]:
norm = pd.DataFrame(adata.layers["scVI_normalized"], index=adata.obs_names, columns=adata.var_names)
norm = pd.concat([norm, adata.obs[["dataset", "cell_type"]], umap], axis=1)

In [13]:
from pandas.api.types import is_categorical_dtype

log1p = log1p.apply(lambda col: col.astype(str) if is_categorical_dtype(col) else col)
norm = norm.apply(lambda col: col.astype(str) if is_categorical_dtype(col) else col)

C:\Users\gardn\AppData\Local\Temp\ipykernel_31520\815241593.py:3: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  log1p = log1p.apply(lambda col: col.astype(str) if is_categorical_dtype(col) else col)
C:\Users\gardn\AppData\Local\Temp\ipykernel_31520\815241593.py:3: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  log1p = log1p.apply(lambda col: col.astype(str) if is_categorical_dtype(col) else col)
C:\Users\gardn\AppData\Local\Temp\ipykernel_31520\815241593.py:4: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  norm = norm.apply(lambda col: col.astype(str) if is_categorical_dtype(col) else col)
C:\Users\gardn\AppData\Local\Temp\ipykernel_31520\815241593.py:4: DeprecationWarning: is_categor

In [22]:
log1p.dataset = [i.replace(",", "") for i in log1p.dataset]
norm.dataset = [i.replace(",", "") for i in norm.dataset]

In [23]:
log1p.to_hdf("../data/etac_counts.h5", key="data")
norm.to_hdf("../data/etac_scvi.h5", key="data")